In [1]:
import numpy as np
from simnibs import mesh_io
from simnibs.simulation import fem
import subprocess

# 1. Load your mesh
mesh = mesh_io.read_msh("D:/Results/FEM/mesh with electrodes/test_mesh27_2.msh")

# 2. Create conductivity as numpy ndarray
# The array must have one conductivity value for EACH volume element (tetrahedron)
n_elements = mesh.elm.nr  # Number of volume elements
cond = np.zeros(n_elements)  # Initialize array

# 3. Fill conductivity array based on tissue tags
for i in range(n_elements):
    tissue_tag = mesh.elm.tag1[i]  # Get tissue tag for this element
    
    if tissue_tag == 1:   # White matter
        cond[i] = 0.12
    elif tissue_tag == 2: # Gray matter
        cond[i] = 0.25
    elif tissue_tag == 3: # CSF
        cond[i] = 1.79
    elif tissue_tag == 4: # Skull
        cond[i] = 0.013
    elif tissue_tag == 5: # Scalp
        cond[i] = 0.25
    elif tissue_tag == 6: #eye balls
        cond[i] = 1
    elif tissue_tag == 7: #compact bone
        cond[i] = 0.007
    elif tissue_tag == 8: #spongy bone
        cond[i] = 0.03 
    elif tissue_tag == 9: #blood
        cond[i] = 0.65
    elif tissue_tag == 10: #muscle
        cond[i] = 0.3
    elif tissue_tag == 13: #necrosis
        cond[i] = 1
    elif tissue_tag == 15: #edema
        cond[i] = 1
    elif tissue_tag == 17: #enhanced tumor
        cond[i] = 0.24
    elif tissue_tag == 2000: #electrode
        cond[i] = 2
    elif tissue_tag == 2001: #electrode
        cond[i] = 2
    elif tissue_tag == 2400: #gel
        cond[i] = 0.2
    elif tissue_tag == 2401: #gel
        cond[i] = 0.2
    else:
        cond[i] = 0.3  # Default conductivity

electrode_tags = [3000, 3001]  # Your electrode surface tags
voltages = [-1.0, 1.0]         # +1V and -1V

# 5. Use tdcs() with numpy array
S = fem.FEMSystem.tdcs(
    mesh=mesh,
    cond=cond,  # Pass the numpy array directly
    electrode_tags=electrode_tags,
    potentials=voltages
)

# 6. Solve
solution = S.solve()

[ simnibs ]INFO: Assembling FEM Matrx
[ simnibs ]INFO: 33.84s to assemble FEM matrix
[ simnibs ]INFO: Using solver options: -ksp_type cg -ksp_rtol 1e-10 -pc_type hypre -pc_hypre_type boomeramg -pc_hypre_boomeramg_coarsen_type HMIS
[ simnibs ]INFO: Preparing the KSP
[ simnibs ]INFO: Time to prepare the KSP: 2.27s
[ simnibs ]INFO: Solving system 1 of 1
[ simnibs ]INFO: Running PETSc with KSP: cg PC: hypre
[ simnibs ]INFO: Number of iterations: 26 Residual Norm: 4.21e-09
[ simnibs ]INFO: KSP Converged with reason: 2
[ simnibs ]INFO: Time to solve system: 6.05s


In [2]:
solution

array([-0.00297577,  0.07369337, -0.16227983, ...,  1.        ,
        1.        ,  1.        ])

In [3]:
v = mesh.view()
v.show()

In [4]:
# Create temporary mesh with solution as NodeData
mesh_with_solution = mesh

# Add solution as nodal data
potential_data = mesh_io.NodeData(
    solution,
    name='potential',
    mesh=mesh_with_solution
)
mesh_with_solution.nodedata.append(potential_data)
mesh_io.write_msh(mesh_with_solution, "D:/Results/FEM/solution/solution_2.msh")


In [5]:
output = "D:/Results/FEM/solution/solution_2"
result_mesh = "D:/Results/FEM/solution/solution_2.msh"
ref_nifti = "D:/Results/FEM/mesh with electrodes/test_labels27_2.nii.gz"
command = f'msh2nii "{result_mesh}" "{ref_nifti}" "{output}" '
subprocess.run(command)
print("done")

done


In [3]:
import numpy as np
from simnibs import mesh_io
from simnibs.simulation import fem
import subprocess

# 1. Load your mesh
mesh = mesh_io.read_msh("D:/Results/FEM/mesh with electrodes/test_mesh27_2.msh")

n_elements = mesh.elm.nr  
print(f"Total number of elements: {n_elements}")

# Get number of nodes
n_nodes = mesh.nodes.nr
print(f"Number of nodes: {n_nodes}")

# Get number of triangles (surface elements)
n_triangles = sum(mesh.elm.elm_type == 2)  # Type 2 = triangles
print(f"Number of triangles: {n_triangles}")

Total number of elements: 2441812
Number of nodes: 331411
Number of triangles: 519387


In [1]:
from simnibs.simulation import fem
help(fem)

Help on module simnibs.simulation.fem in simnibs.simulation:

NAME
    simnibs.simulation.fem - Functions for assembling and solving FEM systems

CLASSES
    builtins.object
        DirichletBC
        FEMSystem
        dofMap
    
    class DirichletBC(builtins.object)
     |  DirichletBC(nodes, values)
     |  
     |  Class Defining  dirichlet boundary conditions
     |  Attributes
     |  ---------
     |  nodes: list
     |      List of nodes there the BC should be applied
     |  
     |  values: list
     |      Value at each node
     |  
     |  Parameters
     |  ---------
     |  nodes: list
     |      List of nodes there the BC should be applied
     |  
     |  values: list
     |      Value at each node
     |  
     |  Methods defined here:
     |  
     |  __init__(self, nodes, values)
     |      Initialize self.  See help(type(self)) for accurate signature.
     |  
     |  apply(self, A, b, dof_map)
     |      Applies the dirichlet BC to the system
     |      Para

In [2]:
my_options = fem.FEMSystem.tdcs.DEFAULT_SOLVER_OPTIONS.copy()


AttributeError: 'function' object has no attribute 'DEFAULT_SOLVER_OPTIONS'